# MATMED Phase 2: Scaling & Ablation Experiments
> This notebook clones the `phase2` branch, downloads the ZINC dataset, runs ablation studies, and generates NeurIPS-style training plots.

## 1. Setup Environment & Clone Repo

In [ ]:
!pip install -q rdkit-pypi torch transformers numpy pandas matplotlib datasets

!git clone https://github.com/scott2srikanth/matmed.git
%cd matmed
!git checkout phase2
print("\n\n✅ Repository cloned, switched to phase2, and dependencies installed!")

## 2. ZINC Dataset Test

In [ ]:
from utils import get_zinc_sample
# Requesting > 100 samples triggers the HuggingFace datasets download
zinc_subset = get_zinc_sample(200)
print(f"\nSampled {len(zinc_subset)} molecules from zpn/zinc250k:")
print(zinc_subset[:5])

## 3. Run Ablation Experiments
This runs three parallel experiments:
1. Full MATMED
2. No Safety Agent (delta=0)
3. No Reaction Agent (beta=0)

*Note: This will take a few minutes on a Colab T4 GPU.*

In [ ]:
!python ablation_experiment.py

## 4. Plot & Compare Metrics
We'll plot the results from the `ablation_experiment.py` output CSVs.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

dfs = {}
for exp in ['full_matmed', 'no_sagent', 'no_ragent']:
    csv_file = f"metrics_{exp}.csv"
    if os.path.exists(csv_file):
        dfs[exp] = pd.read_csv(csv_file)

if dfs:
    fig, ax = plt.subplots(1, 2, figsize=(15, 6))
    
    # Plot 1: Average Reward
    for exp, df in dfs.items():
        if 'avg_reward' in df.columns:
            ax[0].plot(df['iteration'], df['avg_reward'], label=exp, linewidth=2)
    ax[0].set_title("Ablation: Average Agent Reward")
    ax[0].set_xlabel("Iteration")
    ax[0].set_ylabel("Reward")
    ax[0].legend()
    ax[0].grid(True, alpha=0.3)
    
    # Plot 2: Validity %
    for exp, df in dfs.items():
        if 'valid_pct' in df.columns:
            ax[1].plot(df['iteration'], df['valid_pct'], label=exp, linewidth=2)
    ax[1].set_title("Ablation: SMILES Validity %")
    ax[1].set_xlabel("Iteration")
    ax[1].set_ylabel("% Valid Molecules Created")
    ax[1].legend()
    ax[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig("ablation_comparison.png", dpi=200)
    plt.show()
else:
    print("Metrics CSVs not found. Did the ablation experiment finish successfully?")

## 5. View Full Training Plot

In [ ]:
from plot_metrics import plot_metrics
from IPython.display import Image

if os.path.exists("metrics_full_matmed.csv"):
    plot_metrics(csv_path="metrics_full_matmed.csv", output_path="full_matmed_plot.png")
    display(Image("full_matmed_plot.png"))